# Spec-FastGS — BOSCH Dataset Setup & Inference Sweep Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full Spec-FastGS training sweep on Mip-NeRF 360 scenes inside the BOSCH server environment, reading datasets directly from their source path.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables.
2. **Dataset Verification**: Verifies the presence of the Mip-NeRF 360 dataset root `/home/ghp4hc/datasets/datasets/mipneft360`.
3. **Dataset Validation**: Verifies `images_4` exist for all 9 scenes inside their respective subdirectories (`360_v2` or `360_extra_scenes`).
4. **Run Sweep**: Invokes the `run_mip360.sh` batch script directly with `DATA_ROOT="/home/ghp4hc/datasets/datasets/mipneft360"`, logging outputs to `mip360_images4_run.log`.
5. **Results Aggregation**: Formats and prints the training logs and quantitative metrics (`results_grouped.json`) in a neat table.
6. **Output Archiving**: Zips the results to `spec_fastgs_output_mip360_images4.zip` in the same directory.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config & Kernel Check
Defines paths and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')
REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')
ENV_NAME = 'thesis_env'

print(f'Active Python      : {sys.executable}')
print(f'Active Kernel name : {ENV_NAME}')
print(f'Repository Root    : {REPO_ROOT}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection (A100-80GB) and compiles custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import torch
print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print('rasterizer      : OK')
print('simple-knn      : OK')
print('fused-ssim      : OK')

## c03 — Verify Dataset Path
Checks that the Mip-NeRF 360 source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the datasets are downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Mip-NeRF 360 Dataset Layout
Validates the subdirectories to ensure all scenes and `images_4` exist.

In [ ]:
# ── Verify Mip-NeRF 360 Dataset Layout (images_4, all 9 scenes) ────────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]
IMAGES = "images_4"

print(f"Verifying layouts directly under {src_root} ...")
missing = []
for scene in MIP360_SCENES:
    # Check both 360_v2 and 360_extra_scenes subdirectories
    v2_path = os.path.join(src_root, "360_v2", scene)
    extra_path = os.path.join(src_root, "360_extra_scenes", scene)
    
    if os.path.isdir(v2_path):
        scene_dir = v2_path
    elif os.path.isdir(extra_path):
        scene_dir = extra_path
    else:
        scene_dir = None
        
    if scene_dir is None:
        status = "MISSING (scene folder not found in 360_v2 or 360_extra_scenes)"
        missing.append(scene)
    else:
        images_dir = os.path.join(scene_dir, IMAGES)
        if not os.path.isdir(images_dir):
            status = f"MISSING ({IMAGES} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
            missing.append(scene)
        else:
            n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            status = f"OK ({n_imgs} images)"
            
    print(f"  {scene:<12s} {status}")

print()
if missing:
    print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {IMAGES}: {missing}")
    print("    run_mip360.sh will skip these scenes during the sweep.")
else:
    print(f"✅ All {len(MIP360_SCENES)} scenes are verified and available. Ready for training!")

## c05 — Run Spec-FastGS Sweep (`run_mip360.sh`)
Launches the training sweep using the `thesis_env` virtual environment and system CUDA compiler paths.

In [ ]:
# ── Prepare script inputs ──────────────────────────────────────────────────────
import subprocess
import os
import sys

REPO_ROOT = os.path.join(os.getcwd(), "spec-fastgs")
LOGFILE = os.path.join(os.getcwd(), "mip360_images4_run.log")
DATA_ROOT = "/home/ghp4hc/datasets/datasets/mipneft360"

venv_bin = os.path.dirname(sys.executable)
print(f"Virtual environment bin path: {venv_bin}")

cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home:
    # Auto-load search script
    search_script = 'which nvcc 2>/dev/null || (for init in /etc/profile /etc/profile.d/modules.sh; do [ -f "$init" ] && source "$init"; done && for mod in cuda/11.7 cuda/11.8 cuda/12.1 cuda/12.6; do module load "$mod" 2>/dev/null; done && which nvcc 2>/dev/null)'
    r_nvcc = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True)
    if r_nvcc.returncode == 0 and r_nvcc.stdout.strip():
        cuda_home = os.path.dirname(os.path.dirname(r_nvcc.stdout.strip()))
    else:
        cuda_home = '/usr/local/cuda'

print(f"CUDA_HOME path: {cuda_home}")

In [ ]:
%%bash -s "$venv_bin" "$cuda_home" "$LOGFILE" "$REPO_ROOT" "$DATA_ROOT"
ENV_BIN=$1
CUDA_HOME=$2
LOGFILE=$3
REPO_ROOT=$4
export DATA_ROOT=$5

export PATH=$ENV_BIN:$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd "$REPO_ROOT"

echo "Running run_mip360.sh (all 9 scenes directly from ${DATA_ROOT}, images_4) ..."
bash run_mip360.sh > "$LOGFILE" 2>&1
STATUS=$?

echo "--- tail of ${LOGFILE} ---"
tail -n 100 "$LOGFILE"
echo "run_mip360.sh exit status: $STATUS"
if [ $STATUS -ne 0 ]; then
    echo "⚠️  run_mip360.sh stopped early (STOP_ON_ERROR=True) -- check logs above or at ${LOGFILE}"
fi

## c06 — Quantitative Results Summary
Parses `results_grouped.json` and `train_info.json` from the output directory to print a formatted metrics table.

In [ ]:
# ── Quantitative Results Summary ──────────────────────────────────────────────
import json
import os

REPO_ROOT = os.path.join(os.getcwd(), "spec-fastgs")
OUTPUT_ROOT = os.path.join(REPO_ROOT, "output", "mip360_images4")

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

header = f"{'scene':<12s}{'PSNR':>8s}{'SSIM':>8s}{'LPIPS':>8s}{'Spec_PSNR':>11s}{'ASG_IoU':>9s}{'#Gauss':>10s}{'time':>10s}"
print(header)
print("-" * len(header))
for scene in MIP360_SCENES:
    out_dir = os.path.join(OUTPUT_ROOT, scene)
    results_path = os.path.join(out_dir, "results_grouped.json")
    info_path = os.path.join(out_dir, "train_info.json")

    if not os.path.exists(results_path):
        print(f"{scene:<12s}  (no results_grouped.json -- skipped, or sweep did not reach this scene)")
        continue

    with open(results_path) as f:
        results = json.load(f)
    scene_result = next(iter(results.values()))
    render_result = next(iter(scene_result.values()))
    main = render_result.get("main_metrics", {})
    aux = render_result.get("aux_metrics", {})

    info = {}
    if os.path.exists(info_path):
        with open(info_path) as f:
            info = json.load(f)

    print(f"{scene:<12s}{fmt(main.get('PSNR')):>8s}{fmt(main.get('SSIM')):>8s}{fmt(main.get('LPIPS')):>8s}"
          f"{fmt(aux.get('Spec_PSNR')):>11s}{fmt(aux.get('ASG_Residual_IoU')):>9s}"
          f"{str(info.get('final_gaussians', '-')):>10s}{str(info.get('training_time_formatted', '-')):>10s}")

## c07 — Packaging Submission ZIP
Zips the outputs directory into `spec_fastgs_output_mip360_images4.zip`.

In [ ]:
# ── Packaging Submission ZIP ──────────────────────────────────────────────────
import shutil
import os

REPO_ROOT = os.path.join(os.getcwd(), "spec-fastgs")
src = os.path.join(REPO_ROOT, "output", "mip360_images4")
out = os.path.join(os.getcwd(), "spec_fastgs_output_mip360_images4")

if os.path.isdir(src):
    shutil.make_archive(out, "zip", src)
    print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
else:
    print("no mip360_images4 output found at", src)